# Lab: Phishing Detection with AI-Driven Security
**Course: Data Security and Integrity**

Each part contains two kinds of cells:
- **Example**: complete code that shows the workflow. Run it, read the comments, and understand *why* each step is done.
- **Exploration**: a task you must solve yourself in the empty cell. Use the example as a reference, but write the code yourself.

Dataset: Vrbančič, G., Fister Jr., I., & Podgorelec, V. (2020). *Datasets for Phishing Websites Detection*. Data in Brief, 33, 106438. The dataset contains features already extracted from URLs, so this lab does not open any phishing website.

## Setup

In [ ]:
# Replace 0 with the last 3 digits of your student ID (example: 25031554027 -> SEED = 27)
SEED = 0
assert SEED != 0, "Replace SEED with the last 3 digits of your student ID"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_small.csv")

def evaluate(y_true, y_pred):
    """Prints the confusion matrix and four metrics, and returns them as a dictionary."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {"TP": tp, "FN": fn, "FP": fp, "TN": tn,
             "accuracy": accuracy_score(y_true, y_pred),
             "precision": precision_score(y_true, y_pred, zero_division=0),
             "recall": recall_score(y_true, y_pred),
             "f1": f1_score(y_true, y_pred)}
    print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")
    print(f"Accuracy={result['accuracy']:.3f}  Precision={result['precision']:.3f}  Recall={result['recall']:.3f}  F1={result['f1']:.3f}")
    return result

print("Data loaded successfully:", df.shape)


## Part A: Exploring the Phishing Data
### Example
Workflow: (1) check the data size and label distribution, (2) compare feature averages between legitimate and phishing sites. A large difference in averages hints at which features are useful for detection.

In [ ]:
# A1. Data size and label distribution (0 = legitimate, 1 = phishing)
print(df.shape)
print(df["phishing"].value_counts())


In [ ]:
# A2. Average of 10 features for each class
example_features = ["length_url", "qty_dot_url", "qty_slash_url", "directory_length", "domain_length",
                "domain_in_ip", "url_shortened", "tls_ssl_certificate", "time_domain_activation", "qty_redirects"]
df.groupby("phishing")[example_features].mean().round(2).T


### Exploration A (required)
This dataset has 111 features (see `df.columns`). Choose **at least three other features** outside the 10 example features that you suspect distinguish phishing sites. **Write down your guess first in the worksheet**, then test it with code. Was your guess correct?

*Hint: the code pattern is the same as cell A2.*

In [ ]:
# Write your code here


## Part B: Classification with Supervised Learning
### Example
Workflow: (1) separate features `X` and label `y`, (2) split into 70% training and 30% test data with `stratify` so class proportions stay the same, (3) train the model on the training data, (4) evaluate on test data the model has never seen, (5) look at the features the model relies on most.

In [ ]:
# B1. Split the data
X = df.drop(columns="phishing")
y = df["phishing"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)


In [ ]:
# B2. Train a Random Forest and evaluate it
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
result_rf = evaluate(y_test, rf.predict(X_test))


In [ ]:
# B3. Most important features according to the model
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.head(10).round(3)


### Exploration B.1 (required)
Is Random Forest always the best? Train **at least one other model** on the same training and test data, then compare its metrics with Random Forest.

*Hint: `from sklearn.tree import DecisionTreeClassifier`. For LogisticRegression, the data needs scaling: `make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))` from `sklearn.pipeline` and `sklearn.preprocessing`. The fit, predict, evaluate flow is the same as cell B2.*

In [ ]:
# Write your code here


### Exploration B.2 (challenge)
A model with 111 features is hard to explain to users. Retrain the Random Forest **using only the 5 most important features** from cell B3. How much F1-score is lost? Is it worth the simplicity?

*Hint: `importance.index[:5]` holds the names of the top 5 features.*

In [ ]:
# Write your code here


## Part C: Testing under Realistic Conditions (Base-Rate Fallacy)
### Example
Workflow: (1) in the real world phishing is far rarer, so we build a test set where phishing is only 1%, (2) the model gives a phishing probability for every site, (3) a site is flagged as phishing if its probability is ≥ the threshold, (4) evaluate the same model again.

In [ ]:
# C1. Realistic test set with 1% phishing
legit_test = X_test[y_test == 0]
phish_test = X_test[y_test == 1]
n_phish = round(len(legit_test) * 0.01 / 0.99)
X_real = pd.concat([legit_test, phish_test.sample(n_phish, random_state=SEED)])
y_real = pd.Series([0] * len(legit_test) + [1] * n_phish, index=X_real.index)
print("Legitimate:", len(legit_test), " Phishing:", n_phish)


In [ ]:
# C2. Evaluate with threshold 0.5
proba = rf.predict_proba(X_real)[:, 1]
threshold = 0.5
evaluate(y_real, (proba >= threshold).astype(int));


### Exploration C.1 (required)
How does precision change when phishing becomes rarer or more common? Repeat the test for **prevalence 0.1%, 1%, 5%, and 20%**. Record precision and recall, then make a **plot of precision against prevalence**.

*Hint: turn the code in C1 into a function with a prevalence parameter, for example `def make_data(prevalence):` that returns `X_real, y_real`. Use `plt.plot(...)` for the plot.*

In [ ]:
# Write your code here


### Exploration C.2 (challenge)
Go back to the 1% phishing data. Find the **smallest threshold** that gives a precision of at least 0.8. What is the recall at that threshold? What does it mean for users?

*Hint: try thresholds with a loop `for t in np.arange(0.5, 1.0, 0.01):`.*

In [ ]:
# Write your code here


## Part D: Anomaly Detection without Labels
### Example
Workflow: (1) Isolation Forest is trained **without labels**, only on features, (2) the model flags the points that are easiest to isolate as anomalies (-1), (3) we convert -1 to 1 (phishing) so we can compare with the true labels.

In [ ]:
# D1. Isolation Forest with contamination 1%
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=SEED)
iso.fit(X_real)
evaluate(y_real, (iso.predict(X_real) == -1).astype(int));


### Exploration D (required)
The example result is far below the Random Forest. Try to **improve** it by choosing at least one strategy:
(a) use only the 10 most important features from cell B3,
(b) change the contamination value,
(c) train Isolation Forest only on legitimate sites from the training data, then test it on `X_real`.
Does the result improve? What do you conclude about anomaly detection for phishing?

In [ ]:
# Write your code here


## Part E: Attacking the Model (Data Poisoning)
### Example
Workflow: (1) copy the training labels, (2) the attacker relabels 30% of phishing samples as legitimate, for example through fake reports, (3) the model is retrained on the poisoned labels, (4) the model is tested on **clean** test data.

In [ ]:
# E1. Poison 30% of phishing labels
y_poison = y_train.copy()
idx = y_poison[y_poison == 1].sample(frac=0.3, random_state=SEED).index
y_poison.loc[idx] = 0
rf_p = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X_train, y_poison)
evaluate(y_test, rf_p.predict(X_test));


### Exploration E.1 (required)
Repeat the example for poisoning levels of **0%, 20%, 40%, and 60%**, then make a **plot of precision and recall against the poisoning level**. At what level would you say the model is no longer usable?

*Hint: wrap the code in E1 in a loop and store the `evaluate` results in a list.*

In [ ]:
# Write your code here


### Exploration E.2 (challenge)
What if the attacker does the **opposite**, relabeling legitimate sites as phishing? **Write your prediction first in the worksheet**, then test it with 20% and 40% poisoning. What is the impact on users?

In [ ]:
# Write your code here


## Bonus F: Evasion
### Example
Workflow: (1) take phishing sites that were detected, (2) the attacker changes the URL length features to the median value of legitimate sites, (3) calculate what percentage now evades detection.

In [ ]:
# F1. Change 6 URL length features
detected = X_test[(y_test == 1) & (rf.predict(X_test) == 1)]
legit_median = X_train[y_train == 0].median()
url_features = ["length_url", "directory_length", "qty_slash_url", "qty_slash_directory", "file_length", "qty_dot_file"]
X_mod = detected.copy()
X_mod[url_features] = legit_median[url_features].values
print(f"{(rf.predict(X_mod) == 0).mean() * 100:.1f}%", "of phishing evades detection")


### Exploration F
Add `time_domain_activation` (domain age) to the features being changed. What percentage evades detection now? Why is this feature much harder for an attacker to change in the real world than URL length?

In [ ]:
# Write your code here
